# 03 · MongoDB Atlas Connection Test

Validates the connection to the MongoDB Atlas cluster using credentials from `config/mongo.yaml`.  
This notebook contains **no pipeline logic** — it is purely a connectivity check.

**What it tests:**
1. YAML config loads correctly
2. MongoClient can reach the Atlas cluster
3. The target database responds to a ping
4. The target collection is accessible

## Section 1 · Setup & Imports

Load standard libraries. `python-dotenv` is used first so any `.env` overrides are applied before reading the YAML.

In [8]:
%pip install -q pyyaml pymongo python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import pathlib
import yaml
from pymongo import MongoClient
from pymongo.errors import ServerSelectionTimeoutError, OperationFailure
from dotenv import load_dotenv

# Load .env from the project root (one level up from notebooks/)
NOTEBOOK_DIR = pathlib.Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
load_dotenv(PROJECT_ROOT / ".env", override=False)

print("Imports OK. Project root resolved.")

Imports OK. Project root resolved.


## Section 2 · Load Config from YAML

Reads `config/mongo.yaml` relative to the project root. No credential values are printed.

In [10]:
CONFIG_PATH = PROJECT_ROOT / "config" / "mongo.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH}")

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    raw_cfg = yaml.safe_load(f)

cfg = raw_cfg["mongo"]
print("config/mongo.yaml loaded successfully.")

config/mongo.yaml loaded successfully.


## Section 3 · Build MongoDB Connection String

The `host` field in `mongo.yaml` already contains the full Atlas SRV connection string,  
so it is used directly as the URI — no reconstruction needed.

In [11]:
# host already holds the full mongodb+srv:// connection string
MONGO_URI = cfg["host"]
print("Connection URI ready.")

Connection URI ready.


## Section 4 · Test Connection

Instantiates `MongoClient` with a 5-second timeout and calls `server_info()` to verify the cluster is reachable.  
A `ServerSelectionTimeoutError` means the cluster is unreachable from this network or the credentials are wrong.

In [12]:
_conn_ok = False
_server_version = None

try:
    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    info = client.server_info()
    _server_version = info.get("version", "unknown")
    _conn_ok = True
    print(f"Connection established. Server version: {_server_version}")
except ServerSelectionTimeoutError as e:
    print(f"[FAIL] Could not reach the cluster within 5 s.")
    print(f"  Reason: {e}")
except Exception as e:
    print(f"[FAIL] Unexpected error: {type(e).__name__}: {e}")

Connection established. Server version: 8.0.20


## Section 5 · Ping Database & Collection

Runs `db.command('ping')` against the configured database and checks whether the target collection exists.

In [13]:
_db_ok  = False
_col_ok = False

if not _conn_ok:
    print("Skipping — connection was not established.")
else:
    database   = cfg["database"]
    collection = cfg["collection"]
    db         = client[database]

    try:
        ping_result = db.command("ping")
        _db_ok = True
        print(f"Database ping: OK")

        collections = db.list_collection_names()
        _col_ok = collection in collections

        if _col_ok:
            print("Target collection found: OK")
        else:
            print("Target collection not found — it will be created on first write.")

    except OperationFailure as e:
        print(f"[FAIL] Database operation failed: {e}")
    except Exception as e:
        print(f"[FAIL] Unexpected error: {type(e).__name__}: {e}")

Database ping: OK
Target collection found: OK


## Section 6 · Summary

Consolidated PASS / FAIL report for each connectivity step.

In [14]:
def _status(ok: bool) -> str:
    return "PASS" if ok else "FAIL"

_yaml_ok = True  # if we reached this cell without raising, YAML loaded fine

checks = [
    ("YAML loaded",            _yaml_ok),
    ("Connection established", _conn_ok),
    ("Database reachable",     _db_ok),
    ("Collection accessible",  _col_ok),
]

print("=" * 40)
print("  MongoDB Atlas Connection Report")
print("=" * 40)
for label, ok in checks:
    print(f"  [{_status(ok)}]  {label}")
print("=" * 40)

if all(ok for _, ok in checks):
    print("  Overall: PASS")
else:
    print("  Overall: FAIL — review the errors above before proceeding.")

  MongoDB Atlas Connection Report
  [PASS]  YAML loaded
  [PASS]  Connection established
  [PASS]  Database reachable
  [PASS]  Collection accessible
  Overall: PASS
